# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN')

In [3]:
#!pip install pandas

In [4]:
import pandas as pd
from pathlib import Path
csv_path = Path('./data/Data.csv') if Path('./data/Data.csv').exists() else Path('./Data.csv')
df = pd.read_csv(csv_path)

In [5]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld,I loved this product. But they only seem to la...


## LLMChain

In [6]:
!pip install -q langchain-community langchain-classic

In [7]:
!pip install -q langchain langchain-openai python-dotenv pandas

In [8]:
%pip install -U langchain langchain-core langchain-openai langchain-classic


Note: you may need to restart the kernel to use updated packages.


In [9]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain

In [10]:
#Replace None by your own value and justify
# Low temperature gives more consistent descriptions.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

In [11]:
prompt = ChatPromptTemplate.from_template(
    "Describe the following product in 3 engaging sentences, including its benefits: {product}"
)

In [12]:

chain = LLMChain(llm=llm, prompt=prompt)

C:\Users\pc\AppData\Local\Temp\ipykernel_2312\546483037.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt)


In [13]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

print("API key configured:", bool(os.getenv("OPENAI_API_KEY")))

API key configured: True


In [14]:
product = "wireless headphones"

response = chain.invoke({"product": product})

print(response["text"])

Experience unparalleled freedom with our wireless headphones, designed to deliver crystal-clear sound without the hassle of tangled cords. Enjoy the perfect blend of comfort and style, featuring cushioned ear cups and a sleek design that makes them ideal for both workouts and daily commutes. With long-lasting battery life and intuitive controls, you'll stay connected to your favorite music and calls all day long, enhancing your audio experience wherever you go.


## SimpleSequentialChain

In [15]:
from langchain_classic.chains import SimpleSequentialChain

In [16]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "Create a memorable brand name for this product: {product}. Return only the brand name."
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [17]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a catchy one-sentence advertising slogan for this brand: {brand}."
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [18]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [19]:
overall_simple_chain.run(product)
# Repeat twice for different products
for product_example in ["smart water bottle", "eco-friendly backpack"]:
    print(product_example, overall_simple_chain.run(product_example))

C:\Users\pc\AppData\Local\Temp\ipykernel_2312\4228073409.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  overall_simple_chain.run(product)




> Entering new SimpleSequentialChain chain...
AeroSound
"AeroSound: Elevate Your Experience with Every Note!"

> Finished chain.


> Entering new SimpleSequentialChain chain...
AquaMind
"AquaMind: Dive into clarity, where hydration meets inspiration!"

> Finished chain.
smart water bottle "AquaMind: Dive into clarity, where hydration meets inspiration!"


> Entering new SimpleSequentialChain chain...
GreenCarry
"GreenCarry: Carry the Future, Sustainably!"

> Finished chain.
eco-friendly backpack "GreenCarry: Carry the Future, Sustainably!"


**Repeat the above twice for different products**

## SequentialChain

In [20]:
from langchain_classic.chains import SequentialChain

In [21]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

first_prompt = ChatPromptTemplate.from_template(
    "Translate this customer review into English. Return only the translation: {Review}"
)
chain_one = LLMChain(llm=llm, prompt=first_prompt, output_key="English_Review")

In [22]:
second_prompt = ChatPromptTemplate.from_template(
    "Summarize this review in one concise sentence: {English_Review}"
)
chain_two = LLMChain(llm=llm, prompt=second_prompt, output_key="summary")

In [23]:
# prompt template 3: translate to english or other language
third_prompt = ChatPromptTemplate.from_template(
    "Identify the language of this original review. Return only the language name: {Review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt, output_key="language")

In [24]:
# prompt template 4: follow up message that take as inputs the two previous prompts' variables
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a polite customer-service follow-up in {language}, based on this review summary: {summary}"
)
chain_four = LLMChain(llm=llm, prompt=fourth_prompt, output_key="followup_message")

In [25]:
# overall_chain: input= Review
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["English_Review", "summary", "language", "followup_message"],
    verbose=True
)

In [26]:
review = str(df.Review.iloc[5])
overall_chain.invoke({"Review": review})
# Repeat twice for different reviews
for idx in [0, 1]:
    print(overall_chain.invoke({"Review": str(df.Review.iloc[idx])}))



> Entering new SequentialChain chain...

> Finished chain.


> Entering new SequentialChain chain...

> Finished chain.
{'Review': 'I ordered a king size set. My only criticism would be that I wish seller would offer the king size set with 4 pillowcases. I separately ordered a two pack of pillowcases so I could have a total of four. When I saw the two packages, it looked like the color did not exactly match. Customer service was excellent about sending me two more pillowcases so I would have four that matched. Excellent! For the cost of these sheets, I am satisfied with the characteristics and coolness of the sheets.', 'English_Review': 'I ordered a king size set. My only criticism would be that I wish the seller would offer the king size set with 4 pillowcases. I separately ordered a two pack of pillowcases so I could have a total of four. When I saw the two packages, it looked like the color did not exactly match. Customer service was excellent about sending me two more pillowcases

**Repeat the above twice for different products or reviews**

## Router Chain

In [27]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [28]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [29]:
from langchain_classic.chains.router import MultiPromptChain
from langchain_classic.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain_core.prompts import PromptTemplate

In [30]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [31]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [32]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [33]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [34]:
destinations_str = "\n".join(f"{p['name']}: {p['description']}" for p in prompt_infos)
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)
router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [35]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

C:\Users\pc\AppData\Local\Temp\ipykernel_2312\3038952769.py:1: LangChainDeprecationWarning: The class `MultiPromptChain` was deprecated in LangChain 0.2.12 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build routing logic with `create_agent` (e.g. with subagents or prompt-selection middleware). See https://docs.langchain.com/oss/python/langchain/agents
  chain = MultiPromptChain(router_chain=router_chain,


In [36]:
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.


'Black body radiation refers to the electromagnetic radiation emitted by an idealized object known as a "black body." A black body is a perfect absorber of all incident radiation, meaning it absorbs all wavelengths of light and does not reflect any. When heated, a black body emits radiation in a characteristic spectrum that depends solely on its temperature.\n\nThe key features of black body radiation include:\n\n1. **Temperature Dependence**: The intensity and wavelength distribution of the emitted radiation depend on the temperature of the black body. As the temperature increases, the peak wavelength of the emitted radiation shifts to shorter wavelengths (this is described by Wien\'s displacement law).\n\n2. **Planck\'s Law**: The spectral distribution of black body radiation is described by Planck\'s law, which quantifies the intensity of radiation emitted at different wavelengths for a given temperature.\n\n3. **Stefan-Boltzmann Law**: The total energy emitted per unit surface area

In [37]:
chain.run("what is 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2 + 2'}
> Finished chain.


'To solve the problem \\(2 + 2\\), we can break it down into its component parts:\n\n1. Identify the numbers involved: We have the numbers 2 and 2.\n2. Understand the operation: The operation we are performing is addition.\n\nNow, we can combine the two numbers:\n\n\\[\n2 + 2 = 4\n\\]\n\nSo, the answer to the question \\(2 + 2\\) is \\(4\\).'

In [ ]:
chain.run("Why does every cell in our body contain DNA?")
# Extra creative routing examples
for question in ["How does photosynthesis convert sunlight into energy?", "Explain binary search in Python."]:
    print(chain.run(question))



> Entering new MultiPromptChain chain...
biology: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


> Entering new MultiPromptChain chain...
biology: {'input': 'How does photosynthesis convert sunlight into energy?'}
> Finished chain.
Photosynthesis is a complex biochemical process that allows plants, algae, and some bacteria to convert sunlight into chemical energy, primarily in the form of glucose. This process occurs mainly in the chloroplasts of plant cells and can be divided into two main stages: the light-dependent reactions and the light-independent reactions (Calvin cycle).

### 1. Light-Dependent Reactions

These reactions occur in the thylakoid membranes of the chloroplasts and require sunlight. Here’s how they work:

- **Photon Absorption**: Chlorophyll and other pigments in the thylakoid membranes absorb light energy, primarily in the blue and red wavelengths. This energy excites electrons in the chlorophyll molecules.

- **Water Splitting**: T

**Repeat the above at least once for different inputs and chains executions - Be creative!**